# CHIIR 2026 Tutorial on Model Search Behaviour

## Indexing (BM25)

### Install python modules

In [1]:
import sys
!{sys.executable} -m pip install ir_datasets pandas opensearch-py

### Load helper modules

In [2]:
import pprint
from tqdm import tqdm

### Create an OpenSearch Client

In [3]:
from opensearchpy import OpenSearch

In [4]:
host = 'localhost'
port = 9200

client = OpenSearch(
    hosts = [{'host': host, 'port': port}],
    http_compress = True,
    use_ssl = False,
    verify_certs = False,
    ssl_assert_hostname = False,
    ssl_show_warn = False
)

In [5]:
pprint.pprint(client.info())

{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'eGHrQd-_TRCFMeuGXEHjLg',
 'name': 'd7c391a16c06',
 'tagline': 'The OpenSearch Project: https://opensearch.org/',
 'version': {'build_date': '2025-10-29T22:22:22.753988939Z',
             'build_hash': '6564992150e26aaa62d4522a220dfff5188aeb88',
             'build_snapshot': False,
             'build_type': 'tar',
             'distribution': 'opensearch',
             'lucene_version': '10.3.1',
             'minimum_index_compatibility_version': '2.0.0',
             'minimum_wire_compatibility_version': '2.19.0',
             'number': '3.3.2'}}


### Index a Corpus for BM25 Model

Note: Every corpus requires a different configuration for indexing.
- We use [beir/scidocs](https://ir-datasets.com/beir.html#beir/scidocs) as an example

In [6]:
import ir_datasets
dataset_name = "beir/scidocs"
dataset = ir_datasets.load(dataset_name)

Index structure

In [8]:
index_name = "scidocs_bm25"
if client.indices.exists(index=index_name):
    client.indices.delete(index=index_name)

In [ ]:
index_body = {
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0
    }
  },
  "mappings": {
    "properties": {
        "docid": { "type": "keyword" },
        "title": { "type": "text" },
        "text": { "type": "text" },
    }
  }
}
response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

{'acknowledged': True, 'index': 'scidocs_bm25', 'shards_acknowledged': True}


Indexing

In [10]:
for doc in tqdm(dataset.docs_iter(), desc="Indexing"):
    doc_body = {
        "docid": doc.doc_id,
        "title": doc.title,
        "text": doc.text
    }
    response = client.index(index=index_name, body=doc_body)

Indexing: 25657it [01:09, 368.43it/s]


#### Search Test

In [11]:
def search(query: str, size: int = 10) -> dict:
    body = {
        "size": size,
        "query": {
            "multi_match": {
                "query": query,
                "fields": ["title^2", "text"] # title gets a boost
            }
        },
    }

    return client.search(index=index_name, body=body)

In [12]:
q = "Ad Hoc Retrieval Experiments Using WordNet"
resp = search(q, size=5)

print(f"\nTop {len(resp['hits']['hits'])} hits for query: {q}\n")
for hit in resp["hits"]["hits"]:
    src = hit["_source"]
    print(f"[{src['docid']}] {src['title'][:50]}... (score={hit['_score']:.2f})")


Top 5 hits for query: Ad Hoc Retrieval Experiments Using WordNet

[0ef311acf523d4d0e2cc5f747a6508af2c89c5f7] LDA-based document models for ad-hoc retrieval... (score=18.32)
[59407446503d49a8cf5f5643b17502835b62f139] Using WordNet to Disambiguate Word Senses for Text... (score=13.98)
[25190bd8bc97c78626f5ca0b6f59cf0360c71b58] Mobile ad hoc networking: imperatives and challeng... (score=13.97)
[006df3db364f2a6d7cc23f46d22cc63081dd70db] Dynamic source routing in ad hoc wireless networks... (score=13.35)
[384f9e49644a16656cd2f46f3d8213bd2f3f0de3] Towards cloud based mobile ad hoc network simulati... (score=13.35)
